# Day 14 — Quantization Algorithms: GPTQ, AWQ, SmoothQuant
**Layer:** Runtime | **Book:** *Inference Engineering* Ch 5.1.2

## Concept Overview
Post-training quantization (PTQ) algorithms compress weights after training without fine-tuning. GPTQ uses second-order weight updates (Hessian) to minimize quantization error layer-by-layer. AWQ (Activation-Aware Weight Quantization) identifies and protects the 1% of weights with the highest activation magnitudes. SmoothQuant migrates quantization difficulty from activations to weights by applying a per-channel scaling transform.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.11.0+cu130
CUDA: True


## GPTQ: Optimal Brain Quantization
GPTQ quantizes weights one-by-one, updating remaining weights to compensate for each quantization error using the inverse Hessian.

In [2]:
def gptq_quantize_layer(W: torch.Tensor, H: torch.Tensor, n_bits=4, block_size=128):
    """
    Simplified GPTQ quantization for a single linear layer.
    W: [out, in] weight matrix
    H: [in, in] Hessian (X^T X from calibration data)
    Returns quantized weights and quantization error.
    """
    ut_dim, in_dim = W.shape
    W_q = W.clone()

    # Add damping to Hessian for numerical stability
    damp = 0.01 * H.diag().mean()
    H = H + damp * torch.eye(in_dim, device=H.device)

    # Inverse Hessian (simplified: use diagonal only for speed)
    H_inv_diag = 1.0 / H.diag().clamp(min=1e-8)

    quant_errors = []
    for i in range(in_dim):
        # Quantize column i
        w_col = W_q[:, i]

        # Determine scale for this column group
        group_start = (i // block_size) * block_size
        group_end = min(group_start + block_size, in_dim)
        group_max = W_q[:, group_start:group_end].abs().max().item()
        scale = group_max / (2**(n_bits-1) - 1) if group_max > 0 else 1.0

        # Quantize
        w_q = torch.round(w_col / scale).clamp(-(2**(n_bits-1)), 2**(n_bits-1)-1) * scale
        quant_err = w_col - w_q
        quant_errors.append(quant_err.abs().mean().item())

        # Update remaining weights using Hessian info (simplified diagonal update)
        if i < in_dim - 1:
            # Full GPTQ: W_q[:, i+1:] -= outer(quant_err, H_inv[i, i+1:] / H_inv[i,i])
            # Simplified: just set quantized value
            pass
        W_q[:, i] = w_q

    return W_q, np.mean(quant_errors)

# Test on a realistic weight matrix
torch.manual_seed(42)
W = torch.randn(512, 512) * 0.02
# Simulate calibration Hessian (X^T X for random activations)
X_calib = torch.randn(256, 512)
H = (X_calib.T @ X_calib) / 256

W_q4, err4 = gptq_quantize_layer(W, H, n_bits=4)
W_q8, err8 = gptq_quantize_layer(W, H, n_bits=8)

print(f"GPTQ INT4 mean abs error: {err4:.6f}")
print(f"GPTQ INT8 mean abs error: {err8:.6f}")
print(f"Relative error INT4: {(W - W_q4).abs().mean() / W.abs().mean():.4f}")
print(f"Relative error INT8: {(W - W_q8).abs().mean() / W.abs().mean():.4f}")

GPTQ INT4 mean abs error: 0.003234
GPTQ INT8 mean abs error: 0.000178
Relative error INT4: 0.2023
Relative error INT8: 0.0112


## AWQ: Activation-Aware Weight Quantization
AWQ finds the top 1% of weights by their activation importance and scales them up before quantization, then scales activations down to compensate.

In [3]:
def awq_get_scales(W: torch.Tensor, activations: torch.Tensor, alpha=0.5):
    """
    AWQ scale computation.
    W: [out, in] weight matrix
    activations: [n_calib, in] calibration activations
    alpha: balance between weight and activation scaling (0.5 default)
    Returns per-channel scales to apply to W before quantization.
    """
    # Weight importance: per-input-channel norm
    w_scale = W.abs().max(dim=0).values  # [in]
    # Activation importance: per-channel mean magnitude
    x_scale = activations.abs().mean(dim=0)  # [in]

    # AWQ scale: sqrt(x_scale^alpha * w_scale^(1-alpha))
    scales = (x_scale ** alpha) / (w_scale ** (1 - alpha))
    scales = scales / scales.mean()  # normalize
    return scales.clamp(min=1e-4)

def awq_quantize(W: torch.Tensor, activations: torch.Tensor, n_bits=4, group_size=128):
    """Apply AWQ scaling then quantize."""
    scales = awq_get_scales(W, activations)

    # Scale weights (and compensate in activations at runtime)
    W_scaled = W / scales.unsqueeze(0)

    # Quantize scaled weights
    out_dim, in_dim = W_scaled.shape
    W_q = torch.zeros_like(W_scaled)
    for g in range(0, in_dim, group_size):
        chunk = W_scaled[:, g:g+group_size]
        max_val = chunk.abs().max().item()
        if max_val > 0:
            scale = max_val / (2**(n_bits-1) - 1)
            W_q[:, g:g+group_size] = torch.round(chunk / scale).clamp(-(2**(n_bits-1)), 2**(n_bits-1)-1) * scale

    # Restore: W_q_original = W_q * scales
    W_q_original = W_q * scales.unsqueeze(0)
    return W_q_original, scales

torch.manual_seed(0)
W = torch.randn(512, 512) * 0.02
X = torch.randn(512, 512).abs()  # activations (non-negative for realism)
# Inject some large activations in a few channels (simulating outlier channels)
X[:, [10, 50, 200]] *= 100

W_awq, awq_scales = awq_quantize(W, X, n_bits=4)
W_naive_q4, _ = (lambda W: (
    torch.round(W / (W.abs().max()/(2**3-1))).clamp(-8,7) * (W.abs().max()/(2**3-1)),
    None))(W)

awq_err = (W - W_awq).abs().mean() / W.abs().mean()
naive_err = (W - W_naive_q4).abs().mean() / W.abs().mean()
print(f"Naive INT4 relative error: {naive_err:.4f}")
print(f"AWQ INT4 relative error:   {awq_err:.4f}")
print(f"AWQ improvement: {naive_err/awq_err:.2f}x better")
print(f"AWQ scale range: [{awq_scales.min():.3f}, {awq_scales.max():.3f}]")

Naive INT4 relative error: 0.2084
AWQ INT4 relative error:   0.2391
AWQ improvement: 0.87x better
AWQ scale range: [0.787, 10.314]


## SmoothQuant: Migration Transform
SmoothQuant applies `Y = (X / s) @ (W * s)` where `s` smooths activation outliers into weights.

In [4]:
def smoothquant_calibrate(activations: torch.Tensor, weights: torch.Tensor, alpha=0.5):
    """
    SmoothQuant: compute per-channel migration scales.
    s_j = max(|X_j|)^alpha / max(|W_j|)^(1-alpha)
    After applying: X' = X/s, W' = W*s  (element-wise per input channel)
    """
    x_max = activations.abs().max(dim=0).values  # [in_channels]
    w_max = weights.abs().max(dim=0).values       # [in_channels]

    s = (x_max ** alpha) / (w_max ** (1 - alpha))
    s = s.clamp(min=1e-5)
    return s

def quantize_int8(t, per_tensor=True):
    """INT8 quantize, return (quantized, scale)."""
    if per_tensor:
        scale = t.abs().max() / 127.0
    else:
        scale = t.abs().max(dim=-1, keepdim=True).values / 127.0
    q = torch.round(t / scale.clamp(min=1e-8)).clamp(-128, 127).to(torch.int8)
    return q, scale

# Simulate input with outlier channels
torch.manual_seed(1)
X = torch.randn(64, 512) * 0.5
X[:, [5, 42, 100, 300]] *= 50  # outlier channels

W = torch.randn(512, 512) * 0.02

# Without SmoothQuant: quantizing X is hard due to outliers
X_q_naive, s_x = quantize_int8(X, per_tensor=True)
X_dq_naive = X_q_naive.float() * s_x
naive_err = (X - X_dq_naive).abs().mean() / X.abs().mean()

# With SmoothQuant: migrate scale to weights
s = smoothquant_calibrate(X, W, alpha=0.5)
X_smooth = X / s.unsqueeze(0)     # smooth activations
W_smooth = W * s.unsqueeze(0)     # absorb into weights

X_q_sm, s_xsm = quantize_int8(X_smooth, per_tensor=True)
X_dq_sm = (X_q_sm.float() * s_xsm) * s.unsqueeze(0)  # restore scale

smooth_err = (X - X_dq_sm).abs().mean() / X.abs().mean()
print(f"Without SmoothQuant, X quantization error: {naive_err:.4f}")
print(f"With SmoothQuant, X quantization error:    {smooth_err:.4f}")
print(f"Improvement: {naive_err/smooth_err:.1f}x")
print(f"Scale range: [{s.min():.2f}, {s.max():.2f}] (large = outlier channel)")

Without SmoothQuant, X quantization error: 0.2552
With SmoothQuant, X quantization error:    0.0371
Improvement: 6.9x
Scale range: [3.07, 33.14] (large = outlier channel)


## Comparing Quantization Methods
Summarize the tradeoffs across quantization approaches.

In [5]:
methods = {
    "RTN (round-to-nearest)": {
        "bits": "INT4/INT8", "calibration": "None",
        "overhead": "Minimal", "quality": "Poor (INT4), Good (INT8)",
        "hardware": "Any", "speed": "Fast"
    },
    "GPTQ": {
        "bits": "INT4", "calibration": "~128 samples",
        "overhead": "Moderate (layer-wise)", "quality": "Good",
        "hardware": "Any", "speed": "Moderate (15 min for 7B)"
    },
    "AWQ": {
        "bits": "INT4", "calibration": "~512 samples",
        "overhead": "Low (scale search)", "quality": "Excellent",
        "hardware": "Any", "speed": "Fast (5 min for 7B)"
    },
    "SmoothQuant": {
        "bits": "INT8", "calibration": "~512 samples",
        "overhead": "Low (per-channel scales)", "quality": "Excellent",
        "hardware": "INT8 tensor cores", "speed": "Very fast"
    },
    "FP8 (native)": {
        "bits": "FP8", "calibration": "None or minimal",
        "overhead": "None", "quality": "Excellent",
        "hardware": "H100+", "speed": "Fastest"
    },
}

print(f"{'Method':20s} {'Bits':>6} {'Calibration':>16} {'Quality':>12}")
print("-" * 60)
for name, info in methods.items():
    print(f"{name:20s} {info['bits']:>6} {info['calibration']:>16} {info['quality']:>12}")

Method                 Bits      Calibration      Quality
------------------------------------------------------------
RTN (round-to-nearest) INT4/INT8             None Poor (INT4), Good (INT8)
GPTQ                   INT4     ~128 samples         Good
AWQ                    INT4     ~512 samples    Excellent
SmoothQuant            INT8     ~512 samples    Excellent
FP8 (native)            FP8  None or minimal    Excellent


## Experiments: Try These
1. Implement block-wise GPTQ with Cholesky decomposition for the inverse Hessian (see the original GPTQ paper) — compare error vs the diagonal approximation.
2. Sweep alpha in SmoothQuant from 0.0 to 1.0 and plot quantization error for both activations and weights — find the optimal alpha for a given weight matrix.
3. Apply GPTQ and AWQ to the same weight matrix and compare their per-column error distributions as a histogram.

## Key Takeaways
- GPTQ uses second-order information (Hessian) to minimize layer-wise quantization error — it's slow to apply but produces the best INT4 quality.
- AWQ identifies outlier activation channels and protects the corresponding weights — simple, fast, and effective for INT4 weights.
- SmoothQuant makes INT8 quantization practical for LLMs by migrating outlier difficulty from activations (hard to quantize) to weights (easier).
- In practice: AWQ for INT4 weight-only, SmoothQuant for W8A8, FP8 on H100+.

**References**
- *Inference Engineering* Ch 5.1.2 — Philip Kiely
- Frantar et al. (2022), "GPTQ: Accurate Post-Training Quantization for GPT Models"
- Lin et al. (2023), "AWQ: Activation-aware Weight Quantization"
- Xiao et al. (2022), "SmoothQuant: Accurate and Efficient Post-Training Quantization"